<a href="https://colab.research.google.com/github/Arhammanj/Arhammanj/blob/main/Optimal_mapping_using_Monte_Carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy plotly

You now have a grid-based city simulation. 30×30 grid → each cell is a location.Warehouse at (0,0). 10 random delivery points. Actions = how the agent moves.Traffic cost = dynamic difficulty per cell move() ensures valid grid moves.Next step in a Monte Carlo

The agent will try to visit all delivery points efficiently using RL.

In [ ]:
import numpy as np
import random
import plotly.graph_objects as go

# Grid size
grid_size = 30

# Start location (warehouse)
start = (0, 0)

# Randomly generate 10 delivery points
num_deliveries = 10
delivery_points = [ (random.randint(0, grid_size-1), random.randint(0, grid_size-1)) for _ in range(num_deliveries) ]

# Actions
actions = ['up', 'down', 'left', 'right']

# Dynamic traffic cost: 1 to 5 units
traffic_grid = np.random.randint(1, 6, size=(grid_size, grid_size))

def move(state, action):
    x, y = state
    if action == 'up': x = max(0, x-1)
    if action == 'down': x = min(grid_size-1, x+1)
    if action == 'left': y = max(0, y-1)
    if action == 'right': y = min(grid_size-1, y+1)
    return (x, y)


.
Moves through the grid using ε-greedy RL policy.
Adds traffic-based negative rewards for moving.
Gives positive reward for deliveries.
Stops when all delivery points are visited.
Returns a Monte Carlo episode to use for value function updates

In [ ]:
def generate_episode_epsilon(value_function, epsilon=0.1):
    state = start
    remaining = delivery_points.copy()
    episode = []

    while remaining:
        # ε-greedy: explore or exploit
        if random.random() < epsilon:
            action = random.choice(actions)
        else:
            # exploit best known action
            best_value = -float('inf')
            best_action = None
            for a in actions:
                next_state = move(state, a)
                value = value_function.get(next_state, 0)
                if value > best_value:
                    best_value = value
                    best_action = a
            action = best_action

        next_state = move(state, action)
        # Reward = negative traffic cost
        reward = -traffic_grid[next_state[0], next_state[1]]
        episode.append((state, reward))
        state = next_state

        # Reward for delivery completion
        if state in remaining:
            reward = 10
            episode.append((state, reward))
            remaining.remove(state)
    return episode

In [ ]:
# -----------------------------
# Generate policy route from Monte Carlo value function
# -----------------------------
def generate_policy_route_mc(V, max_steps=500):
    """
    Generate delivery route based on learned Monte Carlo value function V(s)
    """
    state = start
    route = [state]
    remaining = delivery_points.copy()
    steps = 0
    visited = set()

    while remaining and steps < max_steps:
        steps += 1
        best_value = -float('inf')
        best_action = None

        for action in actions:
            next_state = move(state, action)
            value = V.get(next_state, 0)
            if value > best_value:
                best_value = value
                best_action = action

        if best_action is None:
            # Random move if stuck
            best_action = random.choice(actions)

        state = move(state, best_action)
        route.append(state)

        # Optional: avoid cycling
        visited.add(state)

        # Remove delivered points
        if state in remaining:
            remaining.remove(state)

    return route

# -----------------------------
# Generate route from Monte Carlo
# -----------------------------
route_mc = generate_policy_route_mc(V_mc)
print("Monte Carlo Route Steps:", len(route_mc))
print("Route sample:", route_mc[:10])  # show first 10 steps

In [ ]:
# -----------------------------
#  Monte Carlo route generator
# -----------------------------
def generate_policy_route_mc(V, max_steps=500):
    """
    Generate a delivery route based on learned Monte Carlo value function V(s)
    Args:
        V : dict : value function V(s)
        max_steps : int : safeguard to avoid infinite loops
    Returns:
        route : list of (x, y) tuples
    """
    state = start
    route = [state]
    remaining = delivery_points.copy()
    steps = 0

    while remaining and steps < max_steps:
        steps += 1
        best_value = -float('inf')
        best_action = None

        # ε-greedy like decision: pick best next state based on V
        for action in actions:
            next_state = move(state, action)
            value = V.get(next_state, 0)
            if value > best_value:
                best_value = value
                best_action = action

        if best_action is None:
            # fallback in case V(s) is all zero
            best_action = random.choice(actions)

        state = move(state, best_action)
        route.append(state)

        # remove delivered points
        if state in remaining:
            remaining.remove(state)

    return route

# -----------------------------
# Generate Monte Carlo route
# -----------------------------
route_mc = generate_policy_route_mc(V_mc)
print("Monte Carlo Route Steps:", len(route_mc))
print("Sample of first 10 steps:", route_mc[:10])

Monte Carlo Route Steps: 501
Sample of first 10 steps: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9)]


In [ ]:
# Step 0: Install Folium
!pip install folium

import folium
import random

# -----------------------------
# Step 1: Lahore Map Setup
# -----------------------------
lat_min, lat_max = 31.4, 31.7
lon_min, lon_max = 74.2, 74.5

map_center = [(lat_min + lat_max)/2, (lon_min + lon_max)/2]
city_map = folium.Map(location=map_center, zoom_start=12)

# -----------------------------
# Step 2: Warehouse & Delivery Points
# -----------------------------
num_deliveries = 10
delivery_points_latlon = [(random.uniform(lat_min, lat_max), random.uniform(lon_min, lon_max)) for _ in range(num_deliveries)]
warehouse = (31.55, 74.35)

# Plot warehouse
folium.Marker(warehouse, tooltip="Warehouse", icon=folium.Icon(color='green')).add_to(city_map)

# Plot delivery points
for i, point in enumerate(delivery_points_latlon):
    folium.Marker(point, tooltip=f"Delivery {i+1}", icon=folium.Icon(color='red')).add_to(city_map)

# -----------------------------
# Step 3: Monte Carlo Grid-to-LatLon Conversion
# -----------------------------
grid_size = 30  # your 30x30 grid
# Example route_mc from Monte Carlo RL: list of (x, y) tuples
# route_mc = [(0,0), (0,1), (1,1), ..., (29,29)]  # replace with your actual RL route

# Convert grid coordinates to lat/lon
def grid_to_latlon(grid_coord, grid_size, lat_min, lat_max, lon_min, lon_max):
    x, y = grid_coord
    lat = lat_min + (x / (grid_size-1)) * (lat_max - lat_min)
    lon = lon_min + (y / (grid_size-1)) * (lon_max - lon_min)
    return (lat, lon)

city_route = [grid_to_latlon(s, grid_size, lat_min, lat_max, lon_min, lon_max) for s in route_mc]

# -----------------------------
# Step 4: Plot Monte Carlo Route
# -----------------------------
folium.PolyLine(city_route, color="blue", weight=3, opacity=0.8).add_to(city_map)

# Display interactive map
city_map

In [ ]:
!pip install -q streamlit streamlit-folium folium
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

In [ ]:
%%writefile app.py
import streamlit as st
import folium
from streamlit_folium import st_folium
import random

# -----------------------------
# Parameters
# -----------------------------
grid_size = 30
num_deliveries = 10

# Lahore bounding box
lat_min, lat_max = 31.4, 31.7
lon_min, lon_max = 74.2, 74.5

# Warehouse
warehouse = (31.55, 74.35)

# Generate random delivery points
delivery_points_latlon = [(random.uniform(lat_min, lat_max), random.uniform(lon_min, lon_max)) for _ in range(num_deliveries)]

# Example Monte Carlo route (replace with actual RL output)
route_mc = [(i, i) for i in range(grid_size)]  # simple diagonal demo

def grid_to_latlon(grid_coord, grid_size, lat_min, lat_max, lon_min, lon_max):
    x, y = grid_coord
    lat = lat_min + (x / (grid_size-1)) * (lat_max - lat_min)
    lon = lon_min + (y / (grid_size-1)) * (lon_max - lon_min)
    return (lat, lon)

city_route = [grid_to_latlon(s, grid_size, lat_min, lat_max, lon_min, lon_max) for s in route_mc]

# -----------------------------
# Streamlit Layout
# -----------------------------
st.title("Monte Carlo RL Delivery Route - Lahore")
st.markdown("Interactive web app: warehouse, delivery points, and optimized RL route.")

# Create Folium map
map_center = [(lat_min + lat_max)/2, (lon_min + lon_max)/2]
city_map = folium.Map(location=map_center, zoom_start=12)

# Plot warehouse
folium.Marker(warehouse, tooltip="Warehouse", icon=folium.Icon(color='green')).add_to(city_map)

# Plot delivery points
for i, point in enumerate(delivery_points_latlon):
    folium.Marker(point, tooltip=f"Delivery {i+1}", icon=folium.Icon(color='red')).add_to(city_map)

# Plot Monte Carlo route
folium.PolyLine(city_route, color="blue", weight=3, opacity=0.8).add_to(city_map)

# Display map
st_folium(city_map, width=700, height=500)

Overwriting app.py


In [ ]:
# Run Streamlit in background
get_ipython().system_raw("streamlit run app.py &")

# Create public URL via localtunnel
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦your url is: https://fair-mails-kiss.loca.lt


In [ ]:
# Save Streamlit app to a file
%%writefile app.py
import streamlit as st
import folium
from streamlit_folium import st_folium
import random

# -----------------------------
# Parameters
# -----------------------------
grid_size = 30
num_deliveries = 10

# Lahore bounding box
lat_min, lat_max = 31.4, 31.7
lon_min, lon_max = 74.2, 74.5

# Warehouse
warehouse = (31.55, 74.35)

# Generate random delivery points
delivery_points_latlon = [(random.uniform(lat_min, lat_max), random.uniform(lon_min, lon_max)) for _ in range(num_deliveries)]

# Example Monte Carlo route (replace with your actual RL output)
route_mc = [(i, i) for i in range(grid_size)]  # simple diagonal demo

def grid_to_latlon(grid_coord, grid_size, lat_min, lat_max, lon_min, lon_max):
    x, y = grid_coord
    lat = lat_min + (x / (grid_size-1)) * (lat_max - lat_min)
    lon = lon_min + (y / (grid_size-1)) * (lon_max - lon_min)
    return (lat, lon)

city_route = [grid_to_latlon(s, grid_size, lat_min, lat_max, lon_min, lon_max) for s in route_mc]

# -----------------------------
# Streamlit Layout
# -----------------------------
st.title("Monte Carlo RL Delivery Route - Lahore")
st.markdown("Interactive web app: warehouse, delivery points, and optimized RL route.")

# Create Folium map
map_center = [(lat_min + lat_max)/2, (lon_min + lon_max)/2]
city_map = folium.Map(location=map_center, zoom_start=12)

# Plot warehouse
folium.Marker(warehouse, tooltip="Warehouse", icon=folium.Icon(color='green')).add_to(city_map)

# Plot delivery points
for i, point in enumerate(delivery_points_latlon):
    folium.Marker(point, tooltip=f"Delivery {i+1}", icon=folium.Icon(color='red')).add_to(city_map)

# Plot Monte Carlo route
folium.PolyLine(city_route, color="blue", weight=3, opacity=0.8).add_to(city_map)

# Display interactive map
st_folium(city_map, width=700, height=500)

Writing app.py
